In [1]:
%%writefile train_nwp.py
import os
import math
import torch
import torch.nn as nn
import torch.multiprocessing as mp
from torch.utils.data import DataLoader
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from datasets import load_dataset, DatasetDict
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors
from transformers import PreTrainedTokenizerFast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ==========================================
# 1. SETUP & CONFIG
# ==========================================
class Config:
    # Data & Tokenizer
    data_path = "/kaggle/input/datasets/truongminh3105/nwp-dataset/final_rich_dataset.jsonl"
    vocab_size = 30000
    block_size = 128
    
    # Model Architecture
    embedding_dim = 512
    hidden_dim = 1024
    num_layers = 2
    dropout = 0.2
    
    # Training Loop
    batch_size = 128 # Per GPU batch size
    epochs = 5
    lr = 5e-4
    num_workers = 4
    
    # Distributed Training
    world_size = 2 # 2x GPU T4 trên Kaggle
    
    # Special Tokens
    unk_token = "<unk>"
    pad_token = "<pad>"
    bos_token = "<s>"
    eos_token = "</s>"

config = Config()

# ==========================================
# 2. DATA PIPELINE
# ==========================================
def prepare_data_and_tokenizer():
    print("Loading dataset...")
    # Load raw dataset
    raw_dataset = load_dataset('json', data_files=config.data_path, split='train')
    
    print("Splitting dataset by 'source'...")
    # Chuyển trường 'source' thành ClassLabel để có thể stratify (phân tầng)
    raw_dataset = raw_dataset.class_encode_column("source")
    
    # Chia Train (80%) và Temp (20%) có stratify theo source
    train_temp_split = raw_dataset.train_test_split(test_size=0.2, stratify_by_column="source", seed=42)
    # Chia tiếp Temp thành Val (10%) và Test (10%)
    val_test_split = train_temp_split['test'].train_test_split(test_size=0.5, stratify_by_column="source", seed=42)
    
    dataset = DatasetDict({
        'train': train_temp_split['train'],
        'val': val_test_split['train'],
        'test': val_test_split['test']
    })
    
    print(f"Dataset splits: {dataset}")

    # --- Huấn luyện Byte-Level BPE Tokenizer ---
    print("Training Byte-Level BPE Tokenizer...")
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
    
    trainer = trainers.BpeTrainer(
        vocab_size=config.vocab_size,
        special_tokens=[config.pad_token, config.bos_token, config.eos_token, config.unk_token]
    )
    
    def get_training_corpus():
        for i in range(0, len(dataset['train']), 10000):
            yield dataset['train'][i : i + 10000]["segmented_text"]

    tokenizer.train_from_iterator(get_training_corpus(), trainer)
    
    # Wrap vào PreTrainedTokenizerFast của HF để dễ xài
    fast_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        unk_token=config.unk_token,
        pad_token=config.pad_token,
        bos_token=config.bos_token,
        eos_token=config.eos_token
    )
    
    fast_tokenizer.save_pretrained("./bpe_vi_tokenizer")
    
    # --- Tokenization & Chunking ---
    print("Tokenizing and chunking data...")
    def tokenize_function(examples):
        return fast_tokenizer(examples["segmented_text"], truncation=False)

    tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

    def group_texts(examples):
        # Nối tất cả các token lại
        concatenated = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated[list(examples.keys())[0]])
        # Bỏ đi phần dư không đủ block_size
        total_length = (total_length // config.block_size) * config.block_size
        
        result = {
            k: [t[i : i + config.block_size] for i in range(0, total_length, config.block_size)]
            for k, t in concatenated.items()
        }
        # Tạo label cho Next Word Prediction (dịch sang phải 1 token)
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(group_texts, batched=True, batch_size=1000)
    lm_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    print("Data preparation complete.")
    
    return lm_datasets, fast_tokenizer

# ==========================================
# 3. MODEL DEFINITION
# ==========================================
class GRULanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        

    def forward(self, input_ids, hidden=None):
        embedded = self.dropout(self.embedding(input_ids))
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc(self.dropout(output))
        return logits, hidden

# ==========================================
# 4. DISTRIBUTED TRAINING LOOP (DDP & AMP)
# ==========================================
def ddp_setup(rank, world_size):
    """Cấu hình môi trường phân tán trên Kaggle (1 node, 2 GPUs)"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    torch.cuda.set_device(rank)
    torch.distributed.init_process_group("nccl", rank=rank, world_size=world_size)

def ddp_cleanup():
    torch.distributed.destroy_process_group()

def train_worker(rank, world_size, lm_datasets, pad_idx):
    ddp_setup(rank, world_size)
    
    # Chuẩn bị Data Loader với DistributedSampler
    train_data = lm_datasets['train']
    val_data = lm_datasets['val']
    
    train_sampler = DistributedSampler(train_data, num_replicas=world_size, rank=rank)
    train_loader = DataLoader(
        train_data, 
        batch_size=config.batch_size, 
        sampler=train_sampler, 
        num_workers=config.num_workers,
        pin_memory=True, 
        prefetch_factor=2
    )
    
    val_sampler = DistributedSampler(val_data, num_replicas=world_size, rank=rank, shuffle=False)
    val_loader = DataLoader(val_data, batch_size=config.batch_size, sampler=val_sampler, num_workers=config.num_workers, pin_memory=True)

    # Khởi tạo Model, cấu hình DDP
    model = GRULanguageModel(
        config.vocab_size, config.embedding_dim, config.hidden_dim, 
        config.num_layers, config.dropout, pad_idx
    ).to(rank)
    model = DDP(model, device_ids=[rank])
    
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = AdamW(model.parameters(), lr=config.lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.epochs * len(train_loader))
    
    # Mixed Precision Scaler
    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(config.epochs):
        train_sampler.set_epoch(epoch)
        model.train()
        total_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(rank, non_blocking=True)
            labels = batch['labels'].to(rank, non_blocking=True)
            
            # Shift data: input là [0, n-1], label là [1, n]
            inputs = input_ids[:, :-1]
            targets = labels[:, 1:].contiguous().view(-1)
            
            optimizer.zero_grad()
            
            # Mixed Precision Forward pass
            with torch.amp.autocast('cuda'):
                logits, _ = model(inputs)
                logits = logits.contiguous().view(-1, config.vocab_size)
                loss = criterion(logits, targets)
            
            # Backward pass & Optimize
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            total_loss += loss.item()
            
            if rank == 0 and batch_idx % 100 == 0:
                print(f"Epoch {epoch+1}/{config.epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
        # Validation Logic
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(rank, non_blocking=True)
                labels = batch['labels'].to(rank, non_blocking=True)
                inputs = input_ids[:, :-1]
                targets = labels[:, 1:].contiguous().view(-1)
                
                with torch.amp.autocast('cuda'):
                    logits, _ = model(inputs)
                    logits = logits.contiguous().view(-1, config.vocab_size)
                    loss = criterion(logits, targets)
                    val_loss += loss.item()
                    
        val_loss /= len(val_loader)
        perplexity = math.exp(min(val_loss, 100)) # Tránh overflow
        
        if rank == 0:
            print(f"=== Epoch {epoch+1} Summary ===")
            print(f"Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Perplexity: {perplexity:.4f}")
            
            # Lưu model (chỉ trên Rank 0)
            checkpoint = {
                'model_state_dict': model.module.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_loss': val_loss
            }
            torch.save(checkpoint, f"gru_nwp_epoch_{epoch+1}.pt")
            
    ddp_cleanup()

# Hàm kích hoạt training trong cell Kaggle
def run_training(lm_datasets, pad_idx):
    print("Starting Distributed Training on 2 GPUs...")
    mp.spawn(train_worker, args=(config.world_size, lm_datasets, pad_idx), nprocs=config.world_size, join=True)

# ==========================================
# 5. INFERENCE SCRIPT (BEAM SEARCH)
# ==========================================
def generate_next_words(model, tokenizer, seed_text, max_length=50, beam_width=5, length_penalty=0.7, device='cuda:0'):
    model.eval()
    input_ids = tokenizer.encode(seed_text, return_tensors='pt').to(device)
    
    # Danh sách beam: tuple(score, chuỗi token, hidden_state_hiện_tại)
    beams = [(0.0, input_ids[0].tolist(), None)]
    
    for _ in range(max_length):
        new_beams = []
        for score, seq, hidden in beams:
            # Nếu gặp token kết thúc, đưa thẳng vào new_beams
            if seq[-1] == tokenizer.eos_token_id:
                new_beams.append((score, seq, hidden))
                continue
                
            input_tensor = torch.tensor([seq[-1]]).unsqueeze(0).to(device)
            
            with torch.no_grad():
                logits, next_hidden = model(input_tensor, hidden)
                log_probs = torch.log_softmax(logits[:, -1, :], dim=-1).squeeze(0)
            
            # Lấy top k vocab
            top_log_probs, top_indices = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                token_id = top_indices[i].item()
                token_log_prob = top_log_probs[i].item()
                
                new_seq = seq + [token_id]
                
                # Length penalty: ( (5 + L) / 6 )^alpha
                lp = ((5 + len(new_seq)) / 6) ** length_penalty
                new_score = score + (token_log_prob / lp)
                
                new_beams.append((new_score, new_seq, next_hidden))
                
        # Giữ lại top k beams có score cao nhất
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_width]
        
        # Nếu mọi beam đều đã đạt eos thì dừng sớm
        if all(seq[-1] == tokenizer.eos_token_id for _, seq, _ in beams):
            break
            
    best_seq = beams[0][1]
    return tokenizer.decode(best_seq, skip_special_tokens=True)


if __name__ == '__main__':
    # 1. Pipeline xử lý và huấn luyện Tokenizer
    print(">>> Bước 1: Khởi động Data Pipeline...")
    lm_datasets, fast_tokenizer = prepare_data_and_tokenizer()
    
    # 2. Khởi chạy DDP Training trên 2 GPU T4
    print("\n>>> Bước 2: Bắt đầu quá trình huấn luyện...")
    pad_idx = fast_tokenizer.pad_token_id
    run_training(lm_datasets, pad_idx)
    
    # LƯU Ý VỀ INFERENCE: 
    # Phần code test inference dưới đây chỉ nên được mở comment (bỏ dấu #) 
    # SAU KHI quá trình huấn luyện đã chạy xong và lưu ra file .pt. 
    # Trong lúc đang train, hãy cứ để nó ẩn như thế này.
    
    print("\n--- Testing Inference ---")
    loaded_model = GRULanguageModel(config.vocab_size, config.embedding_dim, config.hidden_dim, config.num_layers, config.dropout, pad_idx).cuda()
    loaded_model.load_state_dict(torch.load("gru_nwp_epoch_5.pt")['model_state_dict'])
    text = generate_next_words(loaded_model, fast_tokenizer, "Hôm nay thời tiết ở thủ đô Hà Nội", max_length=20, beam_width=5)
    print("Generated:", text)


if __name__ == '__main__':
    print(">>> Bước 1: Khởi động Data Pipeline...")
    lm_datasets, fast_tokenizer = prepare_data_and_tokenizer()
    
    print("\n>>> Bước 2: Bắt đầu quá trình huấn luyện...")
    pad_idx = fast_tokenizer.pad_token_id
    run_training(lm_datasets, pad_idx)

Writing train_nwp.py


In [2]:
!python train_nwp.py

>>> Bước 1: Khởi động Data Pipeline...
Loading dataset...
Generating train split: 797461 examples [00:05, 143499.91 examples/s]
Splitting dataset by 'source'...
Casting to class labels: 100%|█| 797461/797461 [00:02<00:00, 368721.20 examples/
Dataset splits: DatasetDict({
    train: Dataset({
        features: ['source', 'domain', 'raw_text', 'segmented_text'],
        num_rows: 637968
    })
    val: Dataset({
        features: ['source', 'domain', 'raw_text', 'segmented_text'],
        num_rows: 79746
    })
    test: Dataset({
        features: ['source', 'domain', 'raw_text', 'segmented_text'],
        num_rows: 79747
    })
})
Training Byte-Level BPE Tokenizer...
[00:00:00] Tokenize words                 ██████████████████ 138910   /   138910[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 138910   /   138910
[00:00:01] Compute merges                 ██████████████████ 29805    /    29805
T